# client

> Client for interacting with the Fewsats API

In [1]:
#| default_exp core

In [2]:
#| export
from fastcore.utils import *
import os
import httpx
from typing import Dict, Any, List
from time import time, sleep

In [3]:
#| hide 
from dotenv import load_dotenv
from fastcore.test import *

In [4]:
#| hide
load_dotenv()

True

The `Client` class handles authentication and provides the foundation for our API interactions.

In [5]:
#| export
class Client:
    "Client for interacting with the Fewsats API"
    def __init__(self,
                 api_key: str = None, # The API key for the Fewsats account
                 base_url: str = "https://hub-5n97k.ondigitalocean.app"): # The Fewsats API base URL
        self.api_key = api_key or os.environ.get("FEWSATS_API_KEY")
        if not self.api_key:
            raise ValueError("The api_key client option must be set either by passing api_key to the client or by setting the FEWSATS_API_KEY environment variable")
        self.base_url = base_url
        self._httpx_client = httpx.Client()
        self._httpx_client.headers.update({"Authorization": f"Token {self.api_key}"})


In [6]:
k = os.getenv("FEWSATS_API_KEY")
fs = Client(api_key=k)

test_eq(fs.api_key, k)
test_eq(fs._httpx_client.headers["Authorization"], f"Token {k}")

## Methods

In [7]:
#| export
@patch
def _request(self: Client, 
             method: str, # The HTTP method to use
             path: str, # The path to request
             **kwargs) -> Dict[str, Any]:
    "Makes an authenticated request to Fewsats API"
    url = f"{self.base_url}/{path}"
    return  self._httpx_client.request(method, url, **kwargs)

In [8]:
r  = fs._request("GET", "v0/users/me")
test_eq(r.status_code, 200)

### User Info

In [9]:
#| export

@patch
def me(self: Client):
    "Retrieve the user's info."
    r = self._request("GET", "v0/users/me")
    r.raise_for_status()
    return r.json()

In [10]:
fs.me()

{'name': 'Fewsats',
 'last_name': 'Tester',
 'email': 'test@fewsats.com',
 'billing_info': None,
 'id': 15,
 'created_at': '2024-12-18T18:19:00.531Z'}

### Balance 

In [11]:
#| export 

@patch
def balance(self: Client):
    "Retrieve the balance of the user's wallet."
    r = self._request("GET", "v0/wallets")
    r.raise_for_status()
    return r.json()

In [12]:
fs.balance()

[{'id': 15, 'balance': 9978, 'currency': 'usd'}]

### Payment Methods

Retrieve the user's payment methods. Useful for checking which card will be used for purchases.

In [13]:
#| export
@patch
def payment_methods(self: Client) -> List[Dict[str, Any]]:
    "Retrieve the user's payment methods, raises an exception for error status codes."
    r = self._request("GET", "v0/stripe/payment-methods")
    r.raise_for_status()
    return r.json()

In [14]:
pm = fs.payment_methods()
pm

[{'id': 5,
  'last4': '4242',
  'brand': 'Visa',
  'exp_month': 12,
  'exp_year': 2034,
  'is_default': True}]

In [15]:
assert isinstance(pm, list)

### Simulate a Purchase

Simulate a purchase and return the resulting state. Useful, for example, to check if a CC charge is needed or the purchase will use the balance.

In [16]:
#| export

@patch
def simulate_payment(self: Client,
                    amount: str): # The amount in USD cents
    "Simulates a purchase, raises an exception for error status codes."
    assert amount.isdigit()
    return self._request("POST", "v0/l402/preview/purchase/amount", json={"amount_usd": amount})


In [17]:
p = fs.simulate_payment(amount="300") # 3.00 USD
p.json()

{'invoice': {'description': 'USD amount preview',
  'amount_usd': 300,
  'amount_btc': 0,
  'macaroon': '',
  'invoice': ''},
 'transaction': {'current_balance': 9978,
  'balance_to_apply': 300,
  'amount_to_charge': 0,
  'final_balance': 9678},
 'already_purchased': False,
 'purchase': None}

In [18]:
#| hide
test_eq(p.status_code, 200)

The following is an example of how to pay a lightning invoice. This is a low level method that should not used by most users. This method will use the default payment method if a charge is needed.

### Pay 

The pay method is asynchronous. It returns the `payment_id` and `status`. Using the `payment_id` we can check the status of the payment.

In [19]:
#| export
@patch
def pay(self:Client,
        purl:str, # payment endpoint URL
        pct:str, # payment context token
        # offer fields
        amount:int, # amount in cents
        balance:int, # balance
        currency:str, # currency
        description:str, # description
        offer_id:str, # offer id
        payment_methods:list[str], # payment methods
        title:str, # offer title
        type:str, # offer type
        pm:str = '', # preferred payment method (optional)
) -> dict: # payment status response
    "POST payment request. Returns payment status response"
    return self._request("POST", "v0/l402/purchases/from-offer", json={
        "payment_request_url": purl,
        "payment_context_token": pct,
        "payment_method": pm,
        "offer": {
            "offer_id": offer_id,
            "title": title,
            "description": description,
            "amount": amount,
            "type": type,
            "currency": currency,
            "balance": balance,
            "payment_methods": payment_methods,
        },
    })

In [20]:
# Example offer from stock.l402.org
ofs = {
   "offers":[
      {
         "amount":1,
         "balance":1,
         "currency":"USD",
         "description":"Purchase 1 credit for API access",
         "offer_id":"offer_c668e0c0",
         "payment_methods":[
            "lightning"
         ],
         "title":"1 Credit Package",
         "type":"top-up"
      },
      {
         "amount":100,
         "balance":120,
         "currency":"USD",
         "description":"Purchase 120 credits for API access",
         "offer_id":"offer_97bf23f7",
         "payment_methods":[
            "lightning",
            "coinbase_commerce"
         ],
         "title":"120 Credits Package",
         "type":"top-up"
      },
      {
         "amount":499,
         "balance":750,
         "currency":"USD",
         "description":"Purchase 750 credits for API access",
         "offer_id":"offer_a896b13c",
         "payment_methods":[
            "lightning",
            "coinbase_commerce",
            "credit_card"
         ],
         "title":"750 Credits Package",
         "type":"top-up"
      }
   ],
   "payment_context_token":"edb53dec-28f5-4cbb-924a-20e9003c20e1",
   "payment_request_url":"https://stock.l402.org/l402/payment-request",
   "terms_url":"https://link-to-terms.com",
   "version":"0.2.1"
}
ofs

{'offers': [{'amount': 1,
   'balance': 1,
   'currency': 'USD',
   'description': 'Purchase 1 credit for API access',
   'offer_id': 'offer_c668e0c0',
   'payment_methods': ['lightning'],
   'title': '1 Credit Package',
   'type': 'top-up'},
  {'amount': 100,
   'balance': 120,
   'currency': 'USD',
   'description': 'Purchase 120 credits for API access',
   'offer_id': 'offer_97bf23f7',
   'payment_methods': ['lightning', 'coinbase_commerce'],
   'title': '120 Credits Package',
   'type': 'top-up'},
  {'amount': 499,
   'balance': 750,
   'currency': 'USD',
   'description': 'Purchase 750 credits for API access',
   'offer_id': 'offer_a896b13c',
   'payment_methods': ['lightning', 'coinbase_commerce', 'credit_card'],
   'title': '750 Credits Package',
   'type': 'top-up'}],
 'payment_context_token': 'edb53dec-28f5-4cbb-924a-20e9003c20e1',
 'payment_request_url': 'https://stock.l402.org/l402/payment-request',
 'terms_url': 'https://link-to-terms.com',
 'version': '0.2.1'}

The pay method allows to specify which payment method to use. If not specified, the backend will decide which payment to use.

In [21]:
r = fs.pay(ofs['payment_request_url'], ofs['payment_context_token'], **ofs['offers'][-1])
r, r.json()

(<Response [200 OK]>,
 {'id': 212,
  'created_at': '2024-12-25T17:08:34.874Z',
  'status': 'success',
  'payment_method': 'lightning'})

Lightning payments have almost instant settlement so often the status will be `success` right away. For credit card payments, we'll have to wait for the stripe payment to settle.

In [22]:
r = fs.pay(ofs['payment_request_url'], ofs['payment_context_token'], **ofs['offers'][-1], pm='credit_card')
r, r.json()

(<Response [200 OK]>,
 {'id': 213,
  'created_at': '2024-12-25T17:08:38.783Z',
  'status': 'pending',
  'payment_method': 'credit_card'})

### Purchase info 

We can check all the details as follows:

In [23]:
#| export
@patch
def payment_info(self:Client,
                  pid:str): # purchase id
    "Retrieve the details of a payment."
    return self._request("GET", f"v0/l402/purchases/{pid}")

In [24]:
pid = r.json()['id']
r = fs.payment_info(pid)
r, r.json()

(<Response [200 OK]>,
 {'id': 213,
  'created_at': '2024-12-25T17:08:38.783Z',
  'status': 'pending',
  'payment_request_url': 'https://stock.l402.org/l402/payment-request',
  'payment_context_token': 'edb53dec-28f5-4cbb-924a-20e9003c20e1',
  'invoice': '',
  'preimage': '',
  'amount': 499,
  'currency': 'usd',
  'payment_method': 'credit_card',
  'title': '750 Credits Package',
  'description': 'Purchase 750 credits for API access',
  'type': 'top-up'})

After the stripe payment settles, the status will be updated to `success`.

In [25]:
#| skip
r = fs.payment_info(pid)
r.json()['status']

'pending'

For convnience we can use the method `wait_for_settlement` to wait for the payment to settle.

In [26]:
#| export
@patch
def wait_for_settlement(self:Client,
                        pid:str, # purchase id
                        max_interval:int=120, # maximum interval between checks in seconds
                        max_wait:int=600): # maximum total wait time in seconds
    "Wait for payment settlement with exponential backoff"
    start,wait = time(),1
    while time() - start < max_wait:
        r = self.payment_info(pid).json()
        status = r['status']
        if status == 'success': return r
        if status == 'failed': raise ValueError(f"Payment {pid} failed")
        sleep(min(wait, max_interval))
        wait *= 2
    raise TimeoutError(f"Payment {pid} did not settle within {max_wait} seconds. Final status: {status}")

In [27]:
#| skip
r = fs.wait_for_settlement(pid)
r

{'id': 213,
 'created_at': '2024-12-25T17:08:38.783Z',
 'status': 'success',
 'payment_request_url': 'https://stock.l402.org/l402/payment-request',
 'payment_context_token': 'edb53dec-28f5-4cbb-924a-20e9003c20e1',
 'invoice': '',
 'preimage': '',
 'amount': 499,
 'currency': 'usd',
 'payment_method': 'credit_card',
 'title': '750 Credits Package',
 'description': 'Purchase 750 credits for API access',
 'type': 'top-up'}

Both the preview and purchase methods automatically use the default payment method if a charge is needed. This client provides a straightforward way to interact with the Fewsats API, making it easy for developers to integrate Fewsats functionality into their applications.

## Agent Demo

We will use [Claudette](https://claudette.answer.ai/) to demonstrate how to pay for content using the Fewsats API.

In [28]:
from claudette import Chat, models

In [29]:
model = models[1]
model

'claude-3-5-sonnet-20240620'

In [30]:
fs.balance()

[{'id': 15, 'balance': 9479, 'currency': 'usd'}]

In [31]:
chat = Chat(model, sp='You are a helpful assistant that can pay offers.', tools=[fs.pay])
pr = f"Could you pay the cheapest offer using lightning {ofs}?"
r = chat.toolloop(pr, trace_func=print)
r

Message(id='msg_012y98EDugr4kodMeQVpcUfg', content=[TextBlock(text='Certainly! I\'ll help you pay for the cheapest offer using Lightning. Let\'s analyze the offers and proceed with the payment.\n\nThe cheapest offer from the provided list is:\n- Amount: 1 cent (USD 0.01)\n- Balance: 1 credit\n- Currency: USD\n- Description: "Purchase 1 credit for API access"\n- Offer ID: offer_c668e0c0\n- Payment Methods: [\'lightning\']\n- Title: "1 Credit Package"\n- Type: top-up\n\nNow, I\'ll use the `pay` function to process this payment. Here\'s the function call:', type='text'), ToolUseBlock(id='toolu_011WVfWGvSc35n6EpLYkJ6JY', input={'purl': 'https://stock.l402.org/l402/payment-request', 'pct': 'edb53dec-28f5-4cbb-924a-20e9003c20e1', 'amount': 1, 'balance': 1, 'currency': 'USD', 'description': 'Purchase 1 credit for API access', 'offer_id': 'offer_c668e0c0', 'payment_methods': ['lightning'], 'title': '1 Credit Package', 'type': 'top-up', 'pm': 'lightning'}, name='pay', type='tool_use')], model='

Great! The payment request has been successfully submitted. The response status code 200 OK indicates that the payment was processed successfully.

To summarize:
1. The cheapest offer (1 cent for 1 credit) was selected.
2. The payment was made using the Lightning network as requested.
3. The payment was successful.

Is there anything else you would like to know about this transaction or any other assistance you need?

<details>

- id: `msg_01VsTrV9TtRyzFRzNb2GDyXQ`
- content: `[{'text': 'Great! The payment request has been successfully submitted. The response status code 200 OK indicates that the payment was processed successfully.\n\nTo summarize:\n1. The cheapest offer (1 cent for 1 credit) was selected.\n2. The payment was made using the Lightning network as requested.\n3. The payment was successful.\n\nIs there anything else you would like to know about this transaction or any other assistance you need?', 'type': 'text'}]`
- model: `claude-3-5-sonnet-20240620`
- role: `assistant`
- stop_reason: `end_turn`
- stop_sequence: `None`
- type: `message`
- usage: `{'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'input_tokens': 1427, 'output_tokens': 97}`

</details>

We can see in the chat history to see that the agent correctlye filled the required information for the payment.

The payment balance has also decreased as expected.

In [32]:
fs.balance(), chat.h

([{'id': 15, 'balance': 9478, 'currency': 'usd'}],
 [{'role': 'user',
   'content': [{'type': 'text',
     'text': "Could you pay the cheapest offer using lightning {'offers': [{'amount': 1, 'balance': 1, 'currency': 'USD', 'description': 'Purchase 1 credit for API access', 'offer_id': 'offer_c668e0c0', 'payment_methods': ['lightning'], 'title': '1 Credit Package', 'type': 'top-up'}, {'amount': 100, 'balance': 120, 'currency': 'USD', 'description': 'Purchase 120 credits for API access', 'offer_id': 'offer_97bf23f7', 'payment_methods': ['lightning', 'coinbase_commerce'], 'title': '120 Credits Package', 'type': 'top-up'}, {'amount': 499, 'balance': 750, 'currency': 'USD', 'description': 'Purchase 750 credits for API access', 'offer_id': 'offer_a896b13c', 'payment_methods': ['lightning', 'coinbase_commerce', 'credit_card'], 'title': '750 Credits Package', 'type': 'top-up'}], 'payment_context_token': 'edb53dec-28f5-4cbb-924a-20e9003c20e1', 'payment_request_url': 'https://stock.l402.org/l40

In [33]:
#|hide
from nbdev.doclinks import nbdev_export
nbdev_export()